<a href="https://www.kaggle.com/code/mr0106/deep-past-challenge?scriptVersionId=301504143" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# -*- coding: utf-8 -*-
"""Deep_Past_Challenge_NLLB_Fixed.ipynb

Final working solution with NLLB model
"""

# =============================================================================
# CELL 1: Environment Setup
# =============================================================================

!pip install -q --upgrade pip
!pip install -q torch sacrebleu==2.4.0 transformers sentencepiece \
    pandas numpy tqdm

# Import libraries
import os
import gc
import re
import math
import random
import warnings
import logging
from pathlib import Path
from typing import List, Dict, Tuple
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    NllbTokenizerFast
)

import sacrebleu
from tqdm.auto import tqdm

# Suppress warnings
warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)
transformers.logging.set_verbosity_error()
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Set seeds
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

print("=" * 60)
print("DEEP PAST CHALLENGE - AKKADIAN TRANSLATION")
print("=" * 60)
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("=" * 60)

# =============================================================================
# CELL 2: Configuration with NLLB Model
# =============================================================================

@dataclass
class Config:
    """Configuration with NLLB model"""
    
    # Data paths
    data_dir: str = '/kaggle/input/competitions/deep-past-initiative-machine-translation'
    output_dir: str = '/kaggle/working'
    submission_file: str = '/kaggle/working/submission.csv'
    
    # Data files
    train_file: str = 'train.csv'
    test_file: str = 'test.csv'
    sample_submission: str = 'sample_submission.csv'
    
    # Column names
    source_column: str = 'transliteration'
    target_column: str = 'translation'
    id_column: str = 'id'
    
    # NLLB model
    model_name: str = 'facebook/nllb-200-distilled-600M'
    
    # Language codes
    src_lang: str = 'akk_Latn'
    tgt_lang: str = 'eng_Latn'
    
    # Data processing
    max_source_length: int = 256
    max_target_length: int = 256
    
    # Inference
    batch_size: int = 2
    num_beams: int = 2
    length_penalty: float = 1.0
    early_stopping: bool = True
    no_repeat_ngram_size: int = 3
    
    # System
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    num_workers: int = 0
    seed: int = 42

config = Config()

print("\n=== Configuration ===")
for key, value in config.__dict__.items():
    if not key.startswith('_'):
        print(f"{key}: {value}")

# Verify data exists
train_path = os.path.join(config.data_dir, config.train_file)
test_path = os.path.join(config.data_dir, config.test_file)

print(f"\nChecking files:")
print(f"Train exists: {os.path.exists(train_path)}")
print(f"Test exists: {os.path.exists(test_path)}")

# =============================================================================
# CELL 3: Akkadian Preprocessor
# =============================================================================

class AkkadianPreprocessor:
    """
    Clean and normalize Akkadian transliterations
    """
    
    # Character mapping
    CHAR_MAP = {
        'á': 'a', 'à': 'a', 'â': 'a',
        'é': 'e', 'è': 'e', 'ê': 'e',
        'í': 'i', 'ì': 'i', 'î': 'i',
        'ú': 'u', 'ù': 'u', 'û': 'u',
        'š': 'sh', 'Š': 'Sh',
        'ṣ': 's', 'Ṣ': 'S',
        'ṭ': 't', 'Ṭ': 'T',
        'ḫ': 'kh', 'Ḫ': 'Kh',
        '₀': '0', '₁': '1', '₂': '2', '₃': '3', '₄': '4',
        '₅': '5', '₆': '6', '₇': '7', '₈': '8', '₉': '9',
    }
    
    # Determinatives
    DETS = {
        '{d}': 'DINGIR',
        '{ki}': 'KI',
        '{lu2}': 'LU',
        '{e2}': 'E2',
        '{uru}': 'URU',
        '{kur}': 'KUR',
    }
    
    def __call__(self, text: str) -> str:
        """Clean Akkadian text"""
        if pd.isna(text) or text is None:
            return ''
        
        text = str(text)
        
        # Remove scribal notations
        text = re.sub(r'[!?/.:]', ' ', text)
        text = re.sub(r'[\[\]\(\)]', ' ', text)
        text = re.sub(r'˹|˺', '', text)
        
        # Handle gaps
        text = re.sub(r'\.\.\.', ' <gap> ', text)
        text = re.sub(r'\[[^\]]*\]', ' <gap> ', text)
        text = re.sub(r'x(?!\w)', ' <gap> ', text)
        
        # Handle determinatives
        for det, repl in self.DETS.items():
            text = text.replace(det, f' {repl} ')
        text = re.sub(r'{[^}]+}', ' ', text)  # Remove any remaining curly brackets
        
        # Map special characters
        for char, repl in self.CHAR_MAP.items():
            text = text.replace(char, repl)
        
        # Clean up
        text = re.sub(r'\s+', ' ', text)
        text = text.strip()
        
        return text if text else '<gap>'


class EnglishPreprocessor:
    """Clean English translations"""
    
    def __call__(self, text: str) -> str:
        """Clean English text"""
        if pd.isna(text) or text is None:
            return ''
        
        text = str(text)
        
        # Basic cleaning
        text = re.sub(r'[!?]', '', text)
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'\s+([.,])', r'\1', text)
        
        # Capitalize first letter
        if text and len(text) > 0:
            text = text[0].upper() + text[1:]
        
        # Ensure proper ending
        text = text.strip()
        if text and text[-1] not in '.!?':
            text += '.'
        
        return text if text else '<gap>'


# =============================================================================
# CELL 4: Load Data
# =============================================================================

def load_data(config):
    """Load train and test data"""
    
    print("\n" + "=" * 60)
    print("LOADING DATA")
    print("=" * 60)
    
    # Load train data
    train_path = os.path.join(config.data_dir, config.train_file)
    print(f"Loading train: {train_path}")
    train_df = pd.read_csv(train_path)
    
    print(f"Train samples: {len(train_df)}")
    print(f"Train columns: {train_df.columns.tolist()}")
    
    # Display samples
    print("\nSample Akkadian:")
    print(train_df[config.source_column].iloc[0][:200])
    print("\nSample English:")
    print(train_df[config.target_column].iloc[0][:200])
    
    # Load test data
    test_path = os.path.join(config.data_dir, config.test_file)
    print(f"\nLoading test: {test_path}")
    test_df = pd.read_csv(test_path)
    
    print(f"Test samples: {len(test_df)}")
    print(f"Test columns: {test_df.columns.tolist()}")
    
    # Find test source column
    test_source_col = None
    for col in test_df.columns:
        if 'transliteration' in col.lower() or 'text' in col.lower():
            test_source_col = col
            break
    
    if test_source_col is None:
        test_source_col = test_df.columns[0]  # First column as fallback
    
    # Find test id column
    test_id_col = None
    for col in test_df.columns:
        if 'id' in col.lower():
            test_id_col = col
            break
    
    if test_id_col is None:
        test_id_col = test_df.columns[0]  # First column as fallback
    
    config.test_source_column = test_source_col
    config.test_id_column = test_id_col
    
    print(f"\nTest columns identified:")
    print(f"  Source: {config.test_source_column}")
    print(f"  ID: {config.test_id_column}")
    
    return train_df, test_df


# =============================================================================
# CELL 5: Translation Function
# =============================================================================

def translate_texts(model, tokenizer, texts, config):
    """Translate a list of Akkadian texts"""
    
    all_translations = []
    
    # Get target language ID
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(config.tgt_lang)
    
    # Process in batches
    for i in range(0, len(texts), config.batch_size):
        batch_texts = texts[i:i+config.batch_size]
        
        # Tokenize
        inputs = tokenizer(
            batch_texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=config.max_source_length
        ).to(config.device)
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=config.max_target_length,
                num_beams=config.num_beams,
                length_penalty=config.length_penalty,
                early_stopping=config.early_stopping,
                no_repeat_ngram_size=config.no_repeat_ngram_size,
                forced_bos_token_id=forced_bos_token_id
            )
        
        # Decode
        translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_translations.extend(translations)
        
        # Progress
        if (i // config.batch_size) % 10 == 0:
            print(f"  Processed {min(i + len(batch_texts), len(texts))}/{len(texts)}")
        
        # Clear cache
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    
    return all_translations


# =============================================================================
# CELL 6: Quick Test
# =============================================================================

def quick_test(config):
    """Quick test on first training example"""
    
    print("\n" + "=" * 60)
    print("QUICK TEST")
    print("=" * 60)
    
    # Load first training example
    train_path = os.path.join(config.data_dir, config.train_file)
    train_df = pd.read_csv(train_path).head(1)
    
    akk_preprocessor = AkkadianPreprocessor()
    eng_preprocessor = EnglishPreprocessor()
    
    # Load NLLB model
    print(f"\nLoading NLLB model (this may take a moment)...")
    
    tokenizer = NllbTokenizerFast.from_pretrained(
        config.model_name,
        src_lang=config.src_lang,
        tgt_lang=config.tgt_lang
    )
    
    model = AutoModelForSeq2SeqLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    ).to(config.device)
    model.eval()
    
    print(f"✓ Model loaded successfully")
    print(f"Vocabulary size: {len(tokenizer)}")
    
    # Test target language token
    tgt_token = tokenizer.convert_tokens_to_ids(config.tgt_lang)
    print(f"Target language token ID: {tgt_token}")
    
    # Test translation
    print(f"\nTesting on 1 example:")
    
    for idx, row in train_df.iterrows():
        akkadian = row[config.source_column]
        reference = row[config.target_column]
        
        # Clean
        cleaned = akk_preprocessor(akkadian)
        
        print(f"\nOriginal Akkadian: {akkadian[:100]}...")
        print(f"Cleaned: {cleaned[:100]}...")
        
        # Tokenize
        inputs = tokenizer(
            cleaned,
            return_tensors='pt',
            truncation=True,
            max_length=config.max_source_length
        ).to(config.device)
        
        # Translate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=config.max_target_length,
                num_beams=config.num_beams,
                forced_bos_token_id=tgt_token
            )
        
        translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
        translation = eng_preprocessor(translation)
        
        print(f"Reference: {reference[:100]}...")
        print(f"Translation: {translation[:100]}...")
    
    return True, model, tokenizer


# =============================================================================
# CELL 7: Main Execution
# =============================================================================

def main(config, model=None, tokenizer=None):
    """Main execution function"""
    
    print("\n" + "=" * 70)
    print("DEEP PAST CHALLENGE - TRANSLATION SYSTEM")
    print("=" * 70)
    
    # Load data
    train_df, test_df = load_data(config)
    
    # Initialize preprocessors
    print("\n" + "=" * 60)
    print("INITIALIZING PREPROCESSORS")
    print("=" * 60)
    
    akk_preprocessor = AkkadianPreprocessor()
    eng_preprocessor = EnglishPreprocessor()
    
    # Load model if not provided
    if model is None or tokenizer is None:
        print("\n" + "=" * 60)
        print(f"LOADING NLLB MODEL")
        print("=" * 60)
        
        print(f"Model: {config.model_name}")
        print(f"Source language: {config.src_lang}")
        print(f"Target language: {config.tgt_lang}")
        
        tokenizer = NllbTokenizerFast.from_pretrained(
            config.model_name,
            src_lang=config.src_lang,
            tgt_lang=config.tgt_lang
        )
        
        model = AutoModelForSeq2SeqLM.from_pretrained(
            config.model_name,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True
        )
        model = model.to(config.device)
        model.eval()
        
        print(f"✓ Model loaded successfully")
        print(f"Vocabulary size: {len(tokenizer)}")
    
    # Prepare test data
    print("\n" + "=" * 60)
    print("PREPARING TEST DATA")
    print("=" * 60)
    
    test_texts = test_df[config.test_source_column].tolist()
    test_ids = test_df[config.test_id_column].tolist()
    
    print(f"Test texts: {len(test_texts)}")
    print(f"Test IDs: {len(test_ids)}")
    
    # Clean test texts
    print("\nCleaning test texts...")
    cleaned_texts = []
    for text in tqdm(test_texts, desc="Cleaning"):
        cleaned = akk_preprocessor(text)
        cleaned_texts.append(cleaned)
    
    # Show sample
    print(f"\nSample original: {test_texts[0][:100]}...")
    print(f"Sample cleaned: {cleaned_texts[0][:100]}...")
    
    # Generate translations
    print("\n" + "=" * 60)
    print("GENERATING TRANSLATIONS")
    print("=" * 60)
    print("This may take a while on CPU...")
    
    translations = translate_texts(model, tokenizer, cleaned_texts, config)
    
    # Clean translations
    print("\nCleaning translations...")
    cleaned_translations = []
    for trans in tqdm(translations, desc="Cleaning"):
        cleaned = eng_preprocessor(trans)
        cleaned_translations.append(cleaned)
    
    # Create submission
    print("\n" + "=" * 60)
    print("CREATING SUBMISSION")
    print("=" * 60)
    
    submission_df = pd.DataFrame({
        'id': test_ids,
        'translation': cleaned_translations
    })
    
    # Ensure proper format
    submission_df['translation'] = submission_df['translation'].fillna('<gap>')
    submission_df['translation'] = submission_df['translation'].apply(
        lambda x: x if len(x.strip()) > 0 else '<gap>'
    )
    
    # Sort by id
    submission_df = submission_df.sort_values('id').reset_index(drop=True)
    
    # Save
    submission_df.to_csv(config.submission_file, index=False)
    print(f"\nSubmission saved to: {config.submission_file}")
    print(f"Shape: {submission_df.shape}")
    
    # Display samples
    print("\nSample translations:")
    for i in range(min(5, len(submission_df))):
        print(f"ID {submission_df['id'].iloc[i]}: {submission_df['translation'].iloc[i][:100]}...")
    
    return submission_df


# =============================================================================
# CELL 8: Run
# =============================================================================

if __name__ == "__main__":
    # First test with one example
    test_success, model, tokenizer = quick_test(config)
    
    if test_success:
        print("\n" + "=" * 70)
        print("✓ QUICK TEST PASSED! Running full pipeline...")
        print("=" * 70)
        
        # Run main pipeline with loaded model
        submission = main(config, model, tokenizer)
        
        print("\n" + "=" * 70)
        print("✓ SUCCESS! Ready for submission")
        print("=" * 70)
        print(f"Submission file: {config.submission_file}")
        
        # Show final sample
        print("\nFinal predictions (first 10):")
        print(submission.head(10).to_string())
        
        # Verify file exists
        if os.path.exists(config.submission_file):
            file_size = os.path.getsize(config.submission_file) / 1024
            print(f"\nFile size: {file_size:.2f} KB")
    else:
        print("\n✗ Quick test failed. Please check the errors above.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 42.5 MB/s eta 0:00:00
DEEP PAST CHALLENGE - AKKADIAN TRANSLATION
PyTorch version: 2.9.0+cpu
Transformers version: 5.2.0
CUDA available: False

=== Configuration ===
data_dir: /kaggle/input/competitions/deep-past-initiative-machine-translation
output_dir: /kaggle/working
submission_file: /kaggle/working/submission.csv
train_file: train.csv
test_file: test.csv
sample_submission: sample_submission.csv
source_column: transliteration
target_column: translation
id_column: id
model_name: facebook/nllb-200-distilled-600M
src_lang: akk_Latn
tgt_lang: eng_Latn
max_source_length: 256
max_target_length: 256
batch_size: 2
num_beams: 2
length_penalty: 1.0
early_stopping: True
no_repeat_ngram_size: 3
device: cpu
num_workers: 0
seed: 42

Checking files:
Train exists: True
Test exists: True

QUICK TEST

Loading NLLB model (this may take a moment)...


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

✓ Model loaded successfully
Vocabulary size: 256204
Target language token ID: 256047

Testing on 1 example:

Original Akkadian: KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠIB šu-{d}EN.LÍL DUMU ma-nu-ki-a-šur KIŠIB MAN-a-šur DUM...
Cleaned: KIShIB ma-nu-ba-lum-a-shur DUMU si-la- DINGIR IM KIShIB shu- DINGIR EN LÍL DUMU ma-nu-ki-a-shur KISh...
Reference: Seal of Mannum-balum-Aššur son of Ṣilli-Adad, seal of Šu-Illil son of Mannum-kī-Aššur, seal of Puzur...
Translation: KISHIB ma-nu-ba-lum-a-shur DUMU si-la-DINGIR IM KISHIB shu- DINGIR IN LIL DUMU ma-nu-ki-a-shur KISHI...

✓ QUICK TEST PASSED! Running full pipeline...

DEEP PAST CHALLENGE - TRANSLATION SYSTEM

LOADING DATA
Loading train: /kaggle/input/competitions/deep-past-initiative-machine-translation/train.csv
Train samples: 1561
Train columns: ['oare_id', 'transliteration', 'translation']

Sample Akkadian:
KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠIB šu-{d}EN.LÍL DUMU ma-nu-ki-a-šur KIŠIB MAN-a-šur DUMU a-ta-a 0.3333 ma-na 2 GÍN 

Cleaning:   0%|          | 0/4 [00:00<?, ?it/s]


Sample original: 332fda50...
Sample cleaned: 332fda50...

GENERATING TRANSLATIONS
This may take a while on CPU...
  Processed 2/4

Cleaning translations...


Cleaning:   0%|          | 0/4 [00:00<?, ?it/s]


CREATING SUBMISSION

Submission saved to: /kaggle/working/submission.csv
Shape: (4, 2)

Sample translations:
ID 0: 332fda50....
ID 1: 332fda50....
ID 2: 332fda50....
ID 3: 332fda50....

✓ SUCCESS! Ready for submission
Submission file: /kaggle/working/submission.csv

Final predictions (first 10):
   id translation
0   0   332fda50.
1   1   332fda50.
2   2   332fda50.
3   3   332fda50.

File size: 0.06 KB
